**Author:** Hui Fang\
**Purpose:** ST 554 Final Project\
Date: 4/20/2026

# Introction

This project demonstrates the end-to-end use of `PySpark` for both machine learning and real-time data processing. The data is adapted from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), which examines how power consumption across different zones of Tetouan city relates to various factors such as time of day, temperature, and humidity.\
The project consistsof two components. The first component focuses on building an **elastic net regression model** using `Spark MLlib`, incorporating SQL-based feature engineering, one-hot encoding, PCA, and cross-validated hyperparameter tuning. The second component extends this work into a streaming context: a `Python` script generates incoming CSV files, `Spark` monitors the folder as a stream, applies the fitted model to produce predictions and residuals in real time, and outputs the results to the console.\
 Together, these components highlight Spark's ability to unify batch modeling and streaming inference within a single, conherent workflow.

# Part I: Fitting Model

First let's import the modules needed and create a Spark session.

In [1]:
# import modules needed
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline

# Create a Spark session
spark = SparkSession.builder.getOrCreate()

## 1. Read in data and convert to Spark DataFrame

In [2]:
# read in data
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_pd.head()
# convert to Spark Dataframe
power_sdf = spark.createDataFrame(power_pd)
# print out the data frame schema
power_sdf.printSchema()
# check the spark DataFrame
power_sdf.show(10)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.0

## 2. Cast `Hour` to Double using SQLTransformer

The printed schema indicates that `Hour` is stored as a long. To ensure compatibility with downstream `MLlib` transformations, we cast `Hour` to `DoubleType` using an `SQLTransformer` within the pipeline.

In [3]:
# cast Hour vaiable to double
hour_cast_sql = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

## 3. Binarize the `Hour` column and One-hot encode the `Month` column

Next, we create a binary indicator for the `Hour` variable, where values less than 6.5 represent nighttime and values greater than or equal to 6.5 represent daytime. Since `Month` is already stored as a numeric type, we can directly apply OneHotEncoder without any additional casting.

In [4]:
# use Binarizer on Hour_double
hour_bin = Binarizer(
    inputCol = "Hour_double",
    outputCol = "Hour_binary",
    threshold = 6.5)

# One-hot encode the month variable
month_encoder = OneHotEncoder(
    inputCols = ["Month"],
    outputCols = ["Month_ohe"])

## 4. Fit a PCA Model

To fit a PCA (Principal Component Analysis), which reduces several correlated variables into a smaller set of summary components, I first assemble `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` into a single feature vector using `VectorAssembler()`. After  fitting the PCA model, Spark produces a PCA transformer that becomes part of the pipeline. In this project, we retain two principal components (PCs).

In [5]:
# assemble variables into a single vector
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows","Diffuse_Flows"],
    outputCol = "pca_input")

# fit a PCA model
pca = PCA(
    k = 2,
    inputCol = "pca_input",
    outputCol = "pca_features")

## 5. Rename response to `label` and assemble the final `features` vector

Next, we add an `SQLTransformer` to rename `Power_Zone_3` as `label`. We then use a final `VectorAssembler` to combine predictors `Hour_binary`, `Power_Zone_1`, `Power_Zone_2`, the one-hot encoded `Month_ohe`, and the two PCA compoents (`pca_features`) into a gingle `features` column for model fitting.

In [6]:
# rename sesponse `Power_Zone_3` to `label`
label_sql = SQLTransformer(
    statement = """
        SELECT *, Power_Zone_3 AS label
        FROM __THIS__
    """
)

# combine preditors into `features`
features_assembler = VectorAssembler(
    inputCols = [
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe",
        "pca_features"],
    outputCol = "features")

## 6. Build the transformation pipeline

Now we chain all transformation stages together into a single Spark MLlib pipeline, ensuring that each transformation is applied in the correct order before model fitting.

In [7]:
# build the pipeline
transform_stages = [
    hour_cast_sql,
    hour_bin,
    month_encoder,
    pca_assembler,
    pca,
    label_sql,
    features_assembler]

transform_pipeline = Pipeline(stages = transform_stages)

# fit the transformation pipeline
transform_model = transform_pipeline.fit(power_sdf)

# transformed data with label + features
train_transformed = transform_model.transform(power_sdf)
train_transformed.select("label", "features").show(5, truncate = False)

+-----------+---------------------------------------------------------------------------------------+
|label      |features                                                                               |
+-----------+---------------------------------------------------------------------------------------+
|20240.96386|(17,[1,2,4,15,16],[34055.6962,16128.87538,1.0,1.7944048636569496,-0.7412746447403988]) |
|20131.08434|(17,[1,2,4,15,16],[29814.68354,19375.07599,1.0,1.8060408300982267,-0.7108534239558009])|
|19668.43373|(17,[1,2,4,15,16],[29128.10127,19006.68693,1.0,1.8102297630563857,-0.7283113191158465])|
|18899.27711|(17,[1,2,4,15,16],[28228.86076,18361.09422,1.0,1.7986676517408797,-0.722091303219947]) |
|18442.40964|(17,[1,2,4,15,16],[27335.6962,17872.34043,1.0,1.8632872016379665,-0.732304664777608])  |
+-----------+---------------------------------------------------------------------------------------+
only showing top 5 rows


The output confirms that the transformation pipeline executed correctly: the response variable has been renamed to `label`, and all predictors have been successfully assembled into the unified `features` vector required for model fitting.

## 7. Fit the Elastic Net model

Once the transformation pipeline has produced the final label and features columns, we proceed to fit the `Elastic Net` regression model. We begin by importing the required modules.

In [8]:
# import modules needed
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml.tuning import CrossValidator

After completing the transformation pipeline, we proceed to fit the `Elastic Net` regression model. This involves instantiating the LinearRegression estimator, constructing the hyperparameter grid for both regularization strength and Elastic Net mixing, defining the RMSE evaluator, configuring the 5-fold CrossValidator, and finally fitting the cross-validated model on the transformed dataset.

In [9]:
# create a regression model instance
lr = LinearRegression(featuresCol = "features", labelCol = "label")

# build the parameter grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .build()
)

# define the evaluator
evaluator = RegressionEvaluator(
    predictionCol = "prediction",
    labelCol = "label",
    metricName = "rmse")

# set up the CrossValidator
cv = CrossValidator(
    estimator = lr,
    estimatorParamMaps = paramGrid,
    evaluator = evaluator,
    numFolds = 5
)

Now we fit the cross-validated `Elastic Net` model on the transformed dataset. This step trains all 121 hyperparameter combinations, evaluates each using 5-fold cross-validation, and automatically selects the model with the lowest RMSE.

In [10]:
# fit the CV model on transformed data
cv_model = cv.fit(train_transformed)

After fitting the model, we can extract the best hyperparameters to inspect the best `Elastic Net` model and report the optimal tuning parameters.

In [11]:
# extract the best model
best_model = cv_model.bestModel

# extract the best tuning parameters
best_reg = best_model._java_obj.getRegParam()
best_alpha = best_model._java_obj.getElasticNetParam()

# print out the best tunning parameters
print("Best regParam (λ):", best_reg)
print("Best elasticNetParam (α):", best_alpha)

# report the CV error (RMSE)
best_cv_rmse = min(cv_model.avgMetrics)
print("Best CV RMSE:", best_cv_rmse)

Best regParam (λ): 0.1
Best elasticNetParam (α): 0.25
Best CV RMSE: 2147.732046665084


The reusults show that the optimal tuning parameters selected by cross-validation are λ = 0.1 and α = 0.25, with the corresponding minimum RMSE of 2147.73.\
Using these best-performing hyperparameters, we can now generate predictions with the fitted Elastic Net model.

In [12]:
# make prediction
train_predictions = best_model.transform(train_transformed)

# extract and print out training RMSE
train_rmse = evaluator.evaluate(train_predictions)
print("Training RMSE:", train_rmse)

Training RMSE: 2147.0973812905327


The trarining set RMSE is 2147.10, which is very close to but slightly lower than the cross-validated RMSE. The small difference indicates that the selected `Elastic Net` model generalizes well and does not exhibit overfitting.

Next, we compute the residuals for the fitted model and present a DataFrame that includes the true `label`, the model's prediction, and the residual (defined as `label - prediction`).

In [13]:
# import module needed
from pyspark.sql.functions import col

# create residual data frame
residuals_df = (
    train_predictions
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)
# show the first 10 rows of the results
residuals_df.show(10, truncate = False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20880.339701300796|-639.3758413007963|
|20131.08434|18659.921478082997|1471.1628619170042|
|19668.43373|18204.427090267316|1464.0066397326846|
|18899.27711|17590.347680929284|1308.9294290707148|
|18442.40964|16996.978968119674|1445.4306718803273|
|18130.12048|16517.366359428463|1612.7541205715388|
|17945.06024|16092.934692963547|1852.1255470364522|
|17459.27711|15722.37908101489 |1736.8980289851097|
|17025.54217|15270.731694848819|1754.810475151182 |
|16794.21687|14938.028946618095|1856.1879233819054|
+-----------+------------------+------------------+
only showing top 10 rows


# Part II: Streaming

For Part II, we construct a streaming workflow by reading .csv files as they arrive in a designated input directory. We generate these files by randomly sampling rows from the full dataset and writing them out sequentially, thereby emulating a real-time data stream. The original dataset can be accessed [here](https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv).

## 1. Read in data and obtain the schema
This step loads the complete dataset once in batch mode so that we can inspect the structure and extract a schema. Structured Streaming requires an explicit schema because Spark cannot infer column types from streaming files.

In [14]:
# import modules needed
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline

# Create a Spark session
spark = SparkSession.builder.getOrCreate()

Read the streaming dataset into pandas and convert it to Spark data frame.

In [15]:
# read the streaming dataset into pandas for instpection
stream_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv")
stream_pd.head()

# convert the pandas data frame to a Spark Dataframe
stream_sdf = spark.createDataFrame(stream_pd)

# print out the data frame schema to verify column names and data types
stream_sdf.printSchema()

# check the Spark DataFrame to ensure correct loading
stream_sdf.show(10)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      4.805|    76.2|     0.081|                0.059|        0.134| 20421.26582| 12908.20669| 14590.84337|    1|   3|
|      4.212|    78.3|     0.081|                0.117|        0.082| 21393.41772| 13575.6

## 2. Set up the schema
Here we extract the schema from the batch DataFrame and use it to configure a streaming reader. This tells `Spark` how to interpret each incoming CSV file and establishes the folder as a live data source.

In [21]:
import os

# Create the directory if it doesn't exist
input_dir = "stream_input"
if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"Created directory: {input_dir}")
else:
    print(f"Directory already exists: {input_dir}")

Directory already exists: stream_input


In [23]:
# extract the schema from the static Spark DataFrame
schema = stream_sdf.schema

# turn the folder into a live data source
# Spark will continuously watch the "stream_input" folder for new CSV files
stream_df = (
    spark.readStream
        .schema(schema)            # extract the schema
        .option("header", True)    # ensure incoming files include a header row
        .csv("stream_input"))       # folder where the .py script will write streaming files

## 3. Transform and Aggregate the streaming data
In this step, we apply the fitted transformation pipeline from Part I to the streaming DataFrame. This produces the same engineered features (including `PCA` output and the `label` column) that were used during model training.

In [27]:
# apply the fitted transformation pipeline from Part I to the stream DataFrame
# This will produce the 'label' and 'features' columns required for prediction
stream_transformed = transform_model.transform(stream_df)

# apply the trained regression model to obtain predictions on the streaming data
pred_df = cv_model.transform(stream_transformed)

## 4. Create residuals from the streaming predictions
In this step, we compute residuals for each incoming record. Residuals quantify the difference between the observed response and the model's predicted value `(label - prediction)`. We then retain only the columns required for output.

In [28]:
# import the column expression helper
from pyspark.sql.functions import col

# compute residuls for each incoming record
pred_df = pred_df.withColumn("residual", col("label") - col("prediction"))

# only keep the target coloumns
pred_df = pred_df.select("label", "prediction", "residual")

## 5. Create the second stream transformation and join the two streaming DataFrames
We will create the second stream transformation of the raw stream. Here, we simply rename the original response variable to `label` so that it matches the transformed stream. We then join the two streaming DataFrames on this shared column.

In [29]:
# rename the response column in the raw stream DataFrame to 'label'
label_df = stream_df.withColumnRenamed("Power_Zone_3", "label")

# join the prediction DataFrame with the renamed raw stream on the 'label'
joined_df = pred_df.join(label_df, on = "label")

## 6. Start the streaming query and write resluts to cosole
Finally, we configure the streaming sink. Using append mode, Spark will print each processed micro-batch to the console as new files arrive in the input directory.

In [30]:
# start the query and write output to the console in append mode
query = (
    joined_df.writeStream
             .outputMode("append")  # append new rows as they arrive
             .format("console")     # print results to the console
             .start()               # start the streaming query
)